# Escolha do modelo a ser usado na aplicação de teste
---

## A) Importando as Bibliotecas e Pacotes a serem usados
---

### 1. Pacotes baixados
---

In [84]:
import pandas as pd
import joblib
from sklearn.metrics import accuracy_score

### 2. Pacotes Predexan
---

In [56]:
import sys
import os

caminho_predexan = os.path.abspath(os.path.join(os.getcwd(), '../..'))

if caminho_predexan not in sys.path:
    sys.path.append(caminho_predexan)

print(f"Pastas encontradas: {os.listdir(caminho_predexan)}")

import predexan.graficos as pg
import predexan.metricas as pm
import predexan.preproc as pp

Pastas encontradas: ['.ipynb_checkpoints', 'analise_exploratoria_pre_processamento', 'dataset', 'imagens', 'modelos', 'predexan', 'README.md']


## B) Determinando os Dataframes à partir das bases de dados pré-processados
---

### 1. Determinando df_treino
---

In [57]:
df_treino = pd.read_parquet('../../dataset/pre_processado/treino_pre_processado.parquet')
df_treino.info()

<class 'pandas.DataFrame'>
Index: 96822 entries, 0 to 175337
Data columns (total 59 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   duracao_conexao_seg                 96822 non-null  float64
 1   num_pacotes_origem                  96822 non-null  float64
 2   num_pacotes_destino                 96822 non-null  float64
 3   total_bytes_origem                  96822 non-null  float64
 4   total_bytes_destino                 96822 non-null  float64
 5   taxa_pacote                         96822 non-null  float64
 6   carga_origem                        96822 non-null  float64
 7   carga_destino                       96822 non-null  float64
 8   pacotes_perdidos_origem             96822 non-null  float64
 9   pacotes_perdidos_destino            96822 non-null  float64
 10  tempo_medio_pacote_origem           96822 non-null  float64
 11  bytes_por_pacote_destino            96822 non-null  floa

### 2. Determinando df_teste
---

In [58]:
df_teste = pd.read_parquet('../../dataset/pre_processado/teste_pre_processado.parquet')
df_teste.info()

<class 'pandas.DataFrame'>
Index: 49971 entries, 0 to 82328
Data columns (total 57 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   duracao_conexao_seg                 49971 non-null  float64
 1   num_pacotes_origem                  49971 non-null  float64
 2   num_pacotes_destino                 49971 non-null  float64
 3   total_bytes_origem                  49971 non-null  float64
 4   total_bytes_destino                 49971 non-null  float64
 5   taxa_pacote                         49971 non-null  float64
 6   carga_origem                        49971 non-null  float64
 7   carga_destino                       49971 non-null  float64
 8   pacotes_perdidos_origem             49971 non-null  float64
 9   pacotes_perdidos_destino            49971 non-null  float64
 10  tempo_medio_pacote_origem           49971 non-null  float64
 11  bytes_por_pacote_destino            49971 non-null  float

## C) Separando os valores em teste e treino
---

Nessa seção, serão separadas as `features` do `target`, tanto para a base de testes quanto de treino.

Há duas possibilidades claras de `target`, presentes nos datasets, dado o cenário de `Classificação` e `Regressão Logística`: `label`e `categoria_ataque.

A primeira retornaria um diagnóstico apontado se há uma probabilidade alta ou baixa de haver um ataque, de maneira geral. Já a segunda, retornaria a probabilidade de haver um ataque específico dentro de suas dez classes.

A escolha do target se embasa tanto na especialização da resposta, quanto no desempenho e custo computacional do modelo. Dessa forma, ao escolher `label`, a resposta é mais simples, mas continua sendo de grande valor, e o custo computacional é muito inferior. Já a escolha de `categoria_ataque`, traz uma resposta mais especializada e de maior qualidade para o usuário, contudo o custo computacional se mostrou alto.

### 1. Atributos do df_treino
---

In [59]:
# Features
X_treino = df_treino.drop(columns=['label_0', 
                                   'label_1',
                                   'categoria_ataque_Analysis',
                                   'categoria_ataque_Backdoor',
                                   'categoria_ataque_DoS',
                                   'categoria_ataque_Exploits',
                                   'categoria_ataque_Fuzzers',
                                   'categoria_ataque_Generic',
                                   'categoria_ataque_Normal',
                                   'categoria_ataque_Reconnaissance',
                                   'categoria_ataque_Shellcode',
                                   'categoria_ataque_Worms'
                                  ])
print(f'O tipo de X_treino é {type(X_treino)}. Possui {X_treino.shape[1]} features e {X_treino.shape[0]} registros.')

# Target
y_treino = df_treino['label_1'] # Escolhendo somente uma, no caso a de ataque, se torna classificação binária
print(f'O tipo de y_treino é {type(y_treino)}')

O tipo de X_treino é <class 'pandas.DataFrame'>. Possui 47 features e 96822 registros.
O tipo de y_treino é <class 'pandas.Series'>


### 2. Atributos do df_teste
---

In [60]:
# Features
X_teste = df_teste.drop(columns=['label_0', 
                                   'label_1',
                                   'categoria_ataque_Analysis',
                                   'categoria_ataque_Backdoor',
                                   'categoria_ataque_DoS',
                                   'categoria_ataque_Exploits',
                                   'categoria_ataque_Fuzzers',
                                   'categoria_ataque_Generic',
                                   'categoria_ataque_Normal',
                                   'categoria_ataque_Reconnaissance',
                                   'categoria_ataque_Shellcode',
                                   'categoria_ataque_Worms'
                                  ])
print(f'O tipo de X_teste é {type(X_teste)}. Possui {X_teste.shape[1]} features e {X_teste.shape[0]} registros.')

# Target
y_teste = df_teste['label_1'] # Escolhendo somente uma, no caso a de ataque, se torna classificação binária
print(f'O tipo de y_teste é {type(y_teste)}')

O tipo de X_teste é <class 'pandas.DataFrame'>. Possui 45 features e 49971 registros.
O tipo de y_teste é <class 'pandas.Series'>


O dataset de teste possui duas features a menos: `servico_conexao_pop3` e `servico_conexao_ssh`. Desta forma, é necessário adicionar ambas, com os valores iguais a `0.0`:

In [61]:
X_teste = X_teste.reindex(columns=X_treino.columns, fill_value=0.0)

print(f'Agora, o tipo de X_teste é {type(X_teste)}. Possui {X_teste.shape[1]} features e {X_teste.shape[0]} registros.')

Agora, o tipo de X_teste é <class 'pandas.DataFrame'>. Possui 47 features e 49971 registros.


## D) Carregando os modelos criados
---

### 1. Regressão Logística
---

In [62]:
rl = joblib.load('../persistencia/modelos_persistentes/regressao_logistica/rl_c.pkl')

### 2. XGBoost
---

In [63]:
xgb = joblib.load('../persistencia/modelos_persistentes/xgboost/xgboost_c.pkl')

### 3. MLP de duas camadas ocultas
---

In [64]:
mlp_2c = joblib.load('../persistencia/modelos_persistentes/mlp/mlp_2c/mlp_2c.pkl')

### 4. MLP de três camadas ocultas
---

In [65]:
mlp_3c = joblib.load('../persistencia/modelos_persistentes/mlp/mlp_3c/mlp_3c.pkl')

### 5. Stacking Ensemble que utiliza MLP de duas camadas ocultas
---

In [66]:
stacking_2c = joblib.load('../persistencia/modelos_persistentes/stacking/stacking_2c.pkl')

### 6. Stacking Ensemble que utiliza MLP de três camadas ocultas
---

In [67]:
stacking_3c = joblib.load('../persistencia/modelos_persistentes/stacking/stacking_3c.pkl')

## E) Métricas dos modelos criados
---

#### 1. Regressão Logística
---

In [80]:
pred_rl = pm.imprimir_desempenho(rl.predict_proba, X_teste)[:, 1]
print(pred_rl)


RELATÓRIO DE DESEMPENHO DE PROCESSAMENTO:

  Execução iniciada em: 10/06/2026 17:26:03

    1. TEMPO: 0.0242s (0.00 min)

    2. MEMÓRIA RAM (Processo Atual):
        - Pico de processamento: 18.68 MB
        - Diferença líquida de RAM no processo: 0.00 MB
        - Total RAM alocada para o processo: 513.93 MB

    3. CPU:
        - Uso de CPU do sistema durante a execução: 40.0%

  Execução concluída em: 10/06/2026 17:26:03

[0.6652835  0.59828585 0.84570639 ... 0.81239789 0.90266011 0.67232225]


In [81]:
_, limiar_rl, _ = pm.extrair_f1(y_teste, pred_rl)
print(limiar_rl)

0.46419322579025546


In [85]:
pred_rl_f1 = (pred_rl > limiar_rl).astype(int)

In [87]:
score_rl = rl.score(X_teste, y_teste)
score_rl_f1 = accuracy_score(y_teste, pred_rl_f1)
print(f'O percentual de acurácia atual do modelo em limiar padrão é de aproximadamente {pm.calcular_percentual(score_rl, 1):.2f}%, enquanto o limiar equilibrado ({limiar_rl}) é de {pm.calcular_percentual(score_rl_f1, 1):.2f}%.')

O percentual de acurácia atual do modelo em limiar padrão é de aproximadamente 67.95%, enquanto o limiar equilibrado (0.46419322579025546) é de 67.93%.


#### 2. XGBoost
---

In [70]:
pred_xgb = pm.imprimir_desempenho(xgb.predict_proba, X_teste)[:, 1]
print(pred_xgb)


RELATÓRIO DE DESEMPENHO DE PROCESSAMENTO:

  Execução iniciada em: 10/06/2026 17:20:19

    1. TEMPO: 0.1465s (0.00 min)

    2. MEMÓRIA RAM (Processo Atual):
        - Pico de processamento: 0.78 MB
        - Diferença líquida de RAM no processo: 0.00 MB
        - Total RAM alocada para o processo: 512.20 MB

    3. CPU:
        - Uso de CPU do sistema durante a execução: 100.0%

  Execução concluída em: 10/06/2026 17:20:19

[0.92691195 0.90430987 0.779801   ... 0.66352606 0.8281838  0.6739468 ]


In [89]:
_, limiar_xgb, _ = pm.extrair_f1(y_teste, pred_xgb)
print(limiar_xgb)

0.8633077


In [90]:
pred_xgb_f1 = (pred_xgb > limiar_xgb).astype(int)

In [96]:
score_xgb = xgb.score(X_teste, y_teste)
score_xgb_f1 = accuracy_score(y_teste, pred_xgb_f1)
print(f'O percentual de acurácia atual do modelo em limiar padrão é de aproximadamente {pm.calcular_percentual(score_xgb, 1):.2f}%, enquanto o limiar equilibrado ({limiar_xgb}) é de {pm.calcular_percentual(score_xgb_f1, 1):.2f}%.')

O percentual de acurácia atual do modelo em limiar padrão é de aproximadamente 78.19%, enquanto o limiar equilibrado (0.8633077144622803) é de 89.87%.


#### 3. MLP de duas camadas ocultas
---

In [92]:
pred_mlp_2c = pm.imprimir_desempenho(mlp_2c.predict_proba, X_teste)[:, 1]
print(pred_mlp_2c)


RELATÓRIO DE DESEMPENHO DE PROCESSAMENTO:

  Execução iniciada em: 10/06/2026 17:32:06

    1. TEMPO: 0.1647s (0.00 min)

    2. MEMÓRIA RAM (Processo Atual):
        - Pico de processamento: 54.52 MB
        - Diferença líquida de RAM no processo: 0.00 MB
        - Total RAM alocada para o processo: 515.74 MB

    3. CPU:
        - Uso de CPU do sistema durante a execução: 27.0%

  Execução concluída em: 10/06/2026 17:32:06

[0.80286941 0.67555343 0.67555348 ... 0.60925381 0.99691326 0.75488699]


In [93]:
_, limiar_mlp_2c, _ = pm.extrair_f1(y_teste, pred_mlp_2c)
print(limiar_mlp_2c)

0.7906421155860779


In [94]:
pred_mlp_2c_f1 = (pred_mlp_2c > limiar_mlp_2c).astype(int)

In [97]:
score_mlp_2c = mlp_2c.score(X_teste, y_teste)
score_mlp_2c_f1 = accuracy_score(y_teste, pred_mlp_2c_f1)
print(f'O percentual de acurácia atual do modelo em limiar padrão é de aproximadamente {pm.calcular_percentual(score_mlp_2c, 1):.2f}%, enquanto o limiar equilibrado ({limiar_mlp_2c}) é de {pm.calcular_percentual(score_mlp_2c_f1, 1):.2f}%.')

O percentual de acurácia atual do modelo em limiar padrão é de aproximadamente 81.17%, enquanto o limiar equilibrado (0.7906421155860779) é de 88.03%.


#### 4. MLP de três camadas ocultas
---

In [99]:
pred_mlp_3c = pm.imprimir_desempenho(mlp_3c.predict_proba, X_teste)[:, 1]
print(pred_mlp_3c)


RELATÓRIO DE DESEMPENHO DE PROCESSAMENTO:

  Execução iniciada em: 10/06/2026 17:34:11

    1. TEMPO: 0.5041s (0.01 min)

    2. MEMÓRIA RAM (Processo Atual):
        - Pico de processamento: 91.12 MB
        - Diferença líquida de RAM no processo: 0.77 MB
        - Total RAM alocada para o processo: 515.94 MB

    3. CPU:
        - Uso de CPU do sistema durante a execução: 22.4%

  Execução concluída em: 10/06/2026 17:34:11

[0.74710584 0.70162752 0.68520092 ... 0.20632736 0.81666134 0.99828674]


In [100]:
_, limiar_mlp_3c, _ = pm.extrair_f1(y_teste, pred_mlp_3c)
print(limiar_mlp_3c)

0.6997834253412136


In [101]:
pred_mlp_3c_f1 = (pred_mlp_3c > limiar_mlp_3c).astype(int)

In [104]:
score_mlp_3c = mlp_3c.score(X_teste, y_teste)
score_mlp_3c_f1 = accuracy_score(y_teste, pred_mlp_3c_f1)
print(f'O percentual de acurácia atual do modelo em limiar padrão é de aproximadamente {pm.calcular_percentual(score_mlp_3c, 1):.2f}%, enquanto o limiar equilibrado ({limiar_mlp_3c}) é de {pm.calcular_percentual(score_mlp_3c_f1, 1):.2f}%.')

O percentual de acurácia atual do modelo em limiar padrão é de aproximadamente 81.55%, enquanto o limiar equilibrado (0.6997834253412136) é de 83.86%.


#### 5. Stacking Ensemble que utiliza MLP de duas camadas ocultas
---

In [107]:
pred_stacking_2c = pm.imprimir_desempenho(stacking_2c.predict_proba, X_teste)[:, 1]
print(pred_stacking_2c)


RELATÓRIO DE DESEMPENHO DE PROCESSAMENTO:

  Execução iniciada em: 10/06/2026 17:37:04

    1. TEMPO: 0.3222s (0.01 min)

    2. MEMÓRIA RAM (Processo Atual):
        - Pico de processamento: 54.92 MB
        - Diferença líquida de RAM no processo: 0.00 MB
        - Total RAM alocada para o processo: 516.90 MB

    3. CPU:
        - Uso de CPU do sistema durante a execução: 71.2%

  Execução concluída em: 10/06/2026 17:37:05

[0.77546835 0.72978223 0.71324782 ... 0.6365375  0.75606213 0.67773956]


In [108]:
_, limiar_stacking_2c, _ = pm.extrair_f1(y_teste, pred_stacking_2c)
print(limiar_stacking_2c)

0.7386380932460161


In [109]:
pred_stacking_2c_f1 = (pred_stacking_2c > limiar_stacking_2c).astype(int)

In [110]:
score_stacking_2c = stacking_2c.score(X_teste, y_teste)
score_stacking_2c_f1 = accuracy_score(y_teste, pred_stacking_2c_f1)
print(f'O percentual de acurácia atual do modelo em limiar padrão é de aproximadamente {pm.calcular_percentual(score_stacking_2c, 1):.2f}%, enquanto o limiar equilibrado ({limiar_stacking_2c}) é de {pm.calcular_percentual(score_stacking_2c_f1, 1):.2f}%.')

O percentual de acurácia atual do modelo em limiar padrão é de aproximadamente 74.94%, enquanto o limiar equilibrado (0.7386380932460161) é de 88.01%.


#### 6. Stacking Ensemble que utiliza MLP de três camadas ocultas
---

In [111]:
pred_stacking_3c = pm.imprimir_desempenho(stacking_3c.predict_proba, X_teste)[:, 1]
print(pred_stacking_3c)


RELATÓRIO DE DESEMPENHO DE PROCESSAMENTO:

  Execução iniciada em: 10/06/2026 17:39:39

    1. TEMPO: 0.5686s (0.01 min)

    2. MEMÓRIA RAM (Processo Atual):
        - Pico de processamento: 91.52 MB
        - Diferença líquida de RAM no processo: 0.76 MB
        - Total RAM alocada para o processo: 517.67 MB

    3. CPU:
        - Uso de CPU do sistema durante a execução: 30.7%

  Execução concluída em: 10/06/2026 17:39:40

[0.81223616 0.79574641 0.67394599 ... 0.56151171 0.71947547 0.52682452]


In [112]:
_, limiar_stacking_3c, _ = pm.extrair_f1(y_teste, pred_stacking_3c)
print(limiar_stacking_3c)

0.7453494268460579


In [113]:
pred_stacking_3c_f1 = (pred_stacking_3c > limiar_stacking_3c).astype(int)

In [114]:
score_stacking_3c = stacking_3c.score(X_teste, y_teste)
score_stacking_3c_f1 = accuracy_score(y_teste, pred_stacking_3c_f1)
print(f'O percentual de acurácia atual do modelo em limiar padrão é de aproximadamente {pm.calcular_percentual(score_stacking_3c, 1):.2f}%, enquanto o limiar equilibrado ({limiar_stacking_3c}) é de {pm.calcular_percentual(score_stacking_3c_f1, 1):.2f}%.')

O percentual de acurácia atual do modelo em limiar padrão é de aproximadamente 83.09%, enquanto o limiar equilibrado (0.7453494268460579) é de 89.73%.


## F) Decisão de escolha
---

O modelo a ser escolhido foi o `Stacking Ensemble que utiliza MLP de três camadas ocultas`. O que embasa a decisão é o fato de ele ter a segunda melhor acurácia (89.73%), abaixo apenas do XGBoost sozinho (89.87%), sendo 0.14% de diferença, apenas, mas com um uso de CPU substancialmente menor durante as predições de teste: 30.7% contra 100% do XGBoost. Os totais de RAM utilizados foram próximos.